# 📘 Preprocessing DaFit App Reviews
Clean and prepare the scraped reviews for sentiment analysis.

In [16]:
!pip install -q textblob Sastrawi emoji nltk langid

In [17]:
import pandas as pd
import re, emoji, string
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import langid
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### 📂 Load the scraped dataset

In [18]:
df = pd.read_csv("scraping_dafit_reviews.csv")
df['content'] = df['content'].fillna('')

Load and ensures no empty entries in the review text column

### 🌍 Filter reviews written in English only

In [19]:
df['lang'] = df['content'].apply(lambda x: langid.classify(x)[0])
df = df[df['lang'] == 'en'].copy()

Keeps only English-language reviews using langid.

### 🔠 Lowercase and clean text

In [20]:
df['content'] = df['content'].str.lower()

def clean_text(text):
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df['content'] = df['content'].apply(clean_text)

Removes emojis, numbers, and punctuation; converts text to lowercase.

### 🧹 Remove stopwords (English + Indonesian)

In [21]:
factory = StopWordRemoverFactory()
stopword_id = factory.create_stop_word_remover()
english_stopwords = set(stopwords.words("english"))

def remove_stopwords(text):
    text = stopword_id.remove(text)
    words = [w for w in text.split() if w not in english_stopwords]
    return ' '.join(words)

df['content_cleaned'] = df['content'].apply(remove_stopwords)

Removes common Indonesian and English stopwords using Sastrawi + NLTK.

### 🪄 Correct spelling using TextBlob

In [22]:
df['content_cleaned'] = df['content_cleaned'].apply(lambda x: str(TextBlob(x).correct()))

Fixes typos using TextBlob.correct().

### 💾 Save the preprocessed data

In [23]:
df.to_csv("preprocessed_dafit_reviews.csv", index=False)
df.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,lang,content_cleaned
0,6de6c806-f71f-42f3-ab64-0a58fec11816,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,good,5,0,v2.8.3-111-geb91bfe647,2025-04-01 05:59:06,NaN,NaN,v2.8.3-111-geb91bfe647,en,good
1,2db5d6c3-0a33-4934-97db-b722f3299d4c,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,good,3,0,v2.8.3-111-geb91bfe647,2025-04-01 04:53:20,NaN,NaN,v2.8.3-111-geb91bfe647,en,good
2,d9227bad-c638-47ea-ba86-502376a410db,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,for what it does the watch is huge and this ap...,1,0,NaN,2025-04-01 04:44:57,NaN,NaN,NaN,en,watch huge pp useless
4,36d9f404-12d1-4c6c-b65d-d8c0895b66f4,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,its just not working for me and its not scanni...,1,0,NaN,2025-03-31 23:06:01,NaN,NaN,NaN,en,working scanning or code like what point scann...
5,5832be7d-109d-4d87-bfce-78dfda3008ee,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,it has been fantastic i do enjoy it,5,0,v2.8.3-111-geb91bfe647,2025-03-31 20:14:52,NaN,NaN,v2.8.3-111-geb91bfe647,en,fantastic enjoy


Computes sentiment polarity/subjectivity and assigns Positive/Neutral/Negative labels based on review score.